In [3]:
import os
import time
import random
import warnings

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms.functional as F
import coremltools as ct

from torch.export import export
from torchvision.io import decode_image

from executorch import version
from executorch.runtime import Runtime
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.apple.coreml.partition import CoreMLPartitioner
from executorch.backends.apple.coreml.compiler import CoreMLBackend

random.seed(0)
warnings.simplefilter("ignore", FutureWarning)
print("Executorch version: ", version.__version__)

Executorch version:  1.1.0


In [4]:
model_constructors = [
    models.efficientnet_v2_s,
]

model_weights = [
    models.EfficientNet_V2_S_Weights.DEFAULT,
]

model_resize_sizes = [
    [384, 384],
]

In [5]:
# Directory to store the exported .pte files
EXPORT_DIR = "exports"

# Suffix for models
SUFFIX = "_coreml_fp32"

# Paths to the exported .pte files
MODEL_PATH: dict[str, str] = {}

def preprocess(img: torch.Tensor, resize_size: list[int]) -> torch.Tensor:
    img = F.resize(img, resize_size, interpolation=F.InterpolationMode.BILINEAR, antialias=True)
    img = F.convert_image_dtype(img, torch.float)
    return img

In [6]:
print("Exporting models...")


class Model(nn.Module):
    def __init__(self, constructor, weights):
        super().__init__()
        self.mean = torch.as_tensor(weights.transforms().mean).view(-1, 1, 1)
        self.std = torch.as_tensor(weights.transforms().std).view(-1, 1, 1)
        self.model = constructor(weights=weights).eval()

    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.model(x)


for constructor, weights, resize_size in zip(model_constructors, model_weights, model_resize_sizes):
    model = Model(constructor, weights)
    inputs = (torch.randn(1, 3, *resize_size),)

    program = to_edge_transform_and_lower(
        export(model, inputs),
        partitioner=[
            CoreMLPartitioner(
                compile_specs=CoreMLBackend.generate_compile_specs(compute_precision=ct.precision.FLOAT32)
            )
        ],
    )
    program = program.to_executorch()

    with open(path := f"{EXPORT_DIR}/{constructor.__name__}{SUFFIX}.pte", "wb") as f:
        MODEL_PATH[constructor.__name__] = path
        f.write(program.buffer)

    print(f"Exported {MODEL_PATH[constructor.__name__]}\n")

Exporting models...


Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 93.46 passes/s]


Exported exports/efficientnet_v2_s_coreml_fp32.pte



In [7]:
print("Verifying models...")

for constructor, weights, resize_size in zip(model_constructors, model_weights, model_resize_sizes):
    print(constructor.__name__.upper())
    path = MODEL_PATH[constructor.__name__]

    # --- ExecuTorch Inference ---
    img = decode_image("assets/dog.png").unsqueeze(0)
    img = preprocess(img, resize_size)
    print(img.shape)

    t = time.perf_counter()
    runtime = Runtime.get()
    program = runtime.load_program(path)
    output, *_ = program.load_method("forward").execute([img])  # type: ignore
    t = time.perf_counter() - t

    print(
        f"Model size: {os.path.getsize(path) / 1024 / 1024:.2f}MB\n"
        f"Run successfully via ExecuTorch in {t:.2f} seconds."
    )

    # --- PyTorch Eager Reference ---
    img = decode_image("assets/dog.png").unsqueeze(0)
    img = F.resize(img, resize_size, interpolation=F.InterpolationMode.BILINEAR, antialias=True)

    ref_model = constructor(weights=weights).eval()
    ref_output = ref_model(weights.transforms()(img))

    torch.testing.assert_close(output, ref_output, rtol=1e-3, atol=1e-4)
    print("Comparison passed!\n")

[program.cpp:154] InternalConsistency verification requested but not available
[ETCoreMLModelManager.mm:478] Cache Miss: Compiled Model with identifier=executorch_15410fdf-ae95-4cb6-b8de-fe7c1cc2cfeb_all was not found in the models cache.
[ETCoreMLModelManager.mm:457] The model in the pte file is not pre-compiled.  Compiling with a 5 min timeout.


Verifying models...
EFFICIENTNET_V2_S
torch.Size([1, 3, 384, 384])


[ETCoreMLModelManager.mm:490] Successfully got compiled model with identifier=executorch_15410fdf-ae95-4cb6-b8de-fe7c1cc2cfeb_all.  Transferring ownership to assetManager.


Model size: 82.53MB
Run successfully via ExecuTorch in 3.43 seconds.
Comparison passed!



In [8]:
img = decode_image("assets/dog.png")

for constructor, weights, resize_size in zip(model_constructors, model_weights, model_resize_sizes):
    img = decode_image("assets/dog.png").unsqueeze(0)
    img = preprocess(img, resize_size)

    path = MODEL_PATH[constructor.__name__]
    runtime = Runtime.get()
    program = runtime.load_program(path)
    output, *_ = program.load_method("forward").execute([img])

    predicted_class = weights.meta["categories"][output.argmax().item()]
    print(f"Predicted class for {constructor.__name__}: {predicted_class}")

[program.cpp:154] InternalConsistency verification requested but not available
[ETCoreMLModelManager.mm:474] Cache Hit: Successfully retrieved compiled model with identifier=executorch_15410fdf-ae95-4cb6-b8de-fe7c1cc2cfeb_all from the models cache.


Predicted class for efficientnet_v2_s: golden retriever
